In [2]:
import pandas as pd

# Load the dataset
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# First look at the data
print("Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

Shape: (7043, 21)

First 5 rows:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# Understanding the dataset
print("Columns:", df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nChurn Distribution:")
print(df['Churn'].value_counts())

Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Data Types:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

Missing Values:
customerID          0
gender       

In [4]:
# Check for hidden missing values
# TotalCharges should be numeric but might have spaces
print("TotalCharges unique problematic values:")
print(df[df['TotalCharges'] == ' ']['TotalCharges'].count())

# Basic statistics
print("\nMonthly Charges Stats:")
print(df['MonthlyCharges'].describe())

print("\nContract Types:")
print(df['Contract'].value_counts())

print("\nInternet Service Types:")
print(df['InternetService'].value_counts())

TotalCharges unique problematic values:
11

Monthly Charges Stats:
count    7043.000000
mean       64.761692
std        30.090047
min        18.250000
25%        35.500000
50%        70.350000
75%        89.850000
max       118.750000
Name: MonthlyCharges, dtype: float64

Contract Types:
Contract
Month-to-month    3875
Two year          1695
One year          1473
Name: count, dtype: int64

Internet Service Types:
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64


In [5]:
# DATA CLEANING

# Fix 1: Convert TotalCharges to numeric (spaces become NaN)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fix 2: Drop the 11 rows with missing TotalCharges
df = df.dropna(subset=['TotalCharges'])

# Fix 3: Drop CustomerID — not useful for prediction
df = df.drop('customerID', axis=1)

# Fix 4: Convert Churn to 1/0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Verify
print("Shape after cleaning:", df.shape)
print("Missing values remaining:", df.isnull().sum().sum())
print("Churn values:", df['Churn'].value_counts())

Shape after cleaning: (7032, 20)
Missing values remaining: 0
Churn values: Churn
0    5163
1    1869
Name: count, dtype: int64


In [6]:
# ENCODING — Convert text to numbers

# These columns only have 2 values (Yes/No) — simple map
binary_columns = ['gender', 'Partner', 'Dependents', 'PhoneService',
                  'PaperlessBilling', 'MultipleLines', 'OnlineSecurity',
                  'OnlineBackup', 'DeviceProtection', 'TechSupport',
                  'StreamingTV', 'StreamingMovies']

for col in binary_columns:
    df[col] = df[col].map({'Yes': 1, 'No': 0, 
                           'Male': 1, 'Female': 0,
                           'No internet service': 0,
                           'No phone service': 0})

# These columns have 3+ values — use get_dummies
df = pd.get_dummies(df, columns=['InternetService', 
                                  'Contract', 
                                  'PaymentMethod'])

# Verify
print("Shape after encoding:", df.shape)
print("Columns:", df.columns.tolist())

Shape after encoding: (7032, 27)
Columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'Churn', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Bank transfer (automatic)', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


In [7]:
# RETENTION ENGINE — Add churn reason based on data patterns
# This powers the personalized offer generation later

def identify_churn_reason(row):
    if row['Contract_Month-to-month'] == 1 and row['MonthlyCharges'] > 70:
        return 'high_bill'
    elif row['Contract_Month-to-month'] == 1 and row['tenure'] < 12:
        return 'new_customer_risk'
    elif row['InternetService_Fiber optic'] == 1 and row['TechSupport'] == 0:
        return 'service_quality'
    elif row['Contract_Month-to-month'] == 1 and row['tenure'] > 24:
        return 'loyalty_risk'
    elif row['OnlineSecurity'] == 0 and row['OnlineBackup'] == 0:
        return 'missing_features'
    else:
        return 'general_risk'

df['churn_reason'] = df.apply(identify_churn_reason, axis=1)

print("Churn Reasons Distribution:")
print(df['churn_reason'].value_counts())

Churn Reasons Distribution:
churn_reason
high_bill            2097
general_risk         1610
missing_features     1451
new_customer_risk    1094
service_quality       459
loyalty_risk          321
Name: count, dtype: int64


In [8]:
# RETENTION OFFERS — Personalized per churn reason

def generate_retention_offer(reason):
    offers = {
        'high_bill': {
            'offer': '20% discount on current plan for 6 months',
            'type': 'Discount',
            'priority': 'High'
        },
        'new_customer_risk': {
            'offer': 'Free premium support + 2 months discount',
            'type': 'Onboarding Support',
            'priority': 'High'
        },
        'service_quality': {
            'offer': 'Free TechSupport upgrade for 3 months',
            'type': 'Service Upgrade',
            'priority': 'Medium'
        },
        'loyalty_risk': {
            'offer': 'Loyalty reward — free device upgrade or 30% off annual plan',
            'type': 'Loyalty Reward',
            'priority': 'High'
        },
        'missing_features': {
            'offer': 'Free OnlineSecurity + OnlineBackup for 3 months',
            'type': 'Feature Unlock',
            'priority': 'Medium'
        },
        'general_risk': {
            'offer': 'Personalized call from retention team',
            'type': 'Human Outreach',
            'priority': 'Low'
        }
    }
    return offers[reason]

# Add offer columns to dataframe
df['retention_offer'] = df['churn_reason'].apply(
    lambda x: generate_retention_offer(x)['offer']
)
df['offer_type'] = df['churn_reason'].apply(
    lambda x: generate_retention_offer(x)['type']
)
df['offer_priority'] = df['churn_reason'].apply(
    lambda x: generate_retention_offer(x)['priority']
)

print("Sample of retention offers:")
print(df[['churn_reason', 'retention_offer', 
          'offer_type', 'offer_priority']].head(10))

Sample of retention offers:
        churn_reason                            retention_offer  \
0  new_customer_risk   Free premium support + 2 months discount   
1       general_risk      Personalized call from retention team   
2  new_customer_risk   Free premium support + 2 months discount   
3       general_risk      Personalized call from retention team   
4          high_bill  20% discount on current plan for 6 months   
5          high_bill  20% discount on current plan for 6 months   
6          high_bill  20% discount on current plan for 6 months   
7  new_customer_risk   Free premium support + 2 months discount   
8          high_bill  20% discount on current plan for 6 months   
9       general_risk      Personalized call from retention team   

           offer_type offer_priority  
0  Onboarding Support           High  
1      Human Outreach            Low  
2  Onboarding Support           High  
3      Human Outreach            Low  
4            Discount           High  


In [9]:
# STEP 1 — Separate features and target
# Drop the non-numeric retention columns we added
X = df.drop(['Churn', 'churn_reason', 'retention_offer', 
             'offer_type', 'offer_priority'], axis=1)
y = df['Churn']

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Churn distribution:\n", y.value_counts())

Feature matrix shape: (7032, 26)
Target shape: (7032,)
Churn distribution:
 Churn
0    5163
1    1869
Name: count, dtype: int64


In [11]:
# STEP 2 — Train/Test Split + SMOTE for class imbalance
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Split first (before SMOTE — important!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Before SMOTE:")
print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Train churn distribution:\n", y_train.value_counts())

# Apply SMOTE only on training data
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE:")
print("X_train:", X_train_sm.shape)
print("Train churn distribution:\n", y_train_sm.value_counts())

Before SMOTE:
X_train: (5625, 26) | X_test: (1407, 26)
Train churn distribution:
 Churn
0    4130
1    1495
Name: count, dtype: int64

After SMOTE:
X_train: (8260, 26)
Train churn distribution:
 Churn
0    4130
1    4130
Name: count, dtype: int64


In [12]:
# STEP 3 — Train Random Forest Model
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1  # use all CPU cores
)

model.fit(X_train_sm, y_train_sm)
print("✅ Model trained successfully!")
print("Number of trees:", model.n_estimators)
print("Features used:", model.n_features_in_)

✅ Model trained successfully!
Number of trees: 100
Features used: 26


In [13]:
# STEP 4 — Evaluate the Model
from sklearn.metrics import (accuracy_score, precision_score, 
                             recall_score, f1_score, 
                             roc_auc_score, classification_report)

# Predict on test data
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Metrics
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
roc_auc   = roc_auc_score(y_test, y_prob)

print("=" * 40)
print("   TeleRetain AI — Model Evaluation")
print("=" * 40)
print(f"Accuracy  : {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")
print("=" * 40)
print("\nDetailed Report:")
print(classification_report(y_test, y_pred, 
      target_names=['No Churn', 'Churn']))

   TeleRetain AI — Model Evaluation
Accuracy  : 0.7662  (76.62%)
Precision : 0.5501
Recall    : 0.6604
F1 Score  : 0.6002
ROC-AUC   : 0.8241

Detailed Report:
              precision    recall  f1-score   support

    No Churn       0.87      0.80      0.83      1033
       Churn       0.55      0.66      0.60       374

    accuracy                           0.77      1407
   macro avg       0.71      0.73      0.72      1407
weighted avg       0.78      0.77      0.77      1407



In [14]:
# STEP 5 — Save the model for FastAPI use later
import joblib
import os

# Save model
joblib.dump(model, 'churn_model.pkl')

# Save feature column names (important for FastAPI)
feature_columns = list(X.columns)
joblib.dump(feature_columns, 'feature_columns.pkl')

print("✅ Model saved: churn_model.pkl")
print("✅ Feature columns saved: feature_columns.pkl")
print(f"Features: {len(feature_columns)} columns")
print("\nColumn names:")
print(feature_columns)

✅ Model saved: churn_model.pkl
✅ Feature columns saved: feature_columns.pkl
Features: 26 columns

Column names:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Bank transfer (automatic)', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']
